In [996]:
import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd
from loguru import logger


Part 1 : We look for molecules with the same id but in different json files

In [997]:
# We load the two main files :
# - entities.tsv: contains the extracted entities
# - grounded_molecules.tsv: conyain the results of the molecules grounding
# - (molecule name,type of database asked, the grounded id etc...)
entities = pd.read_csv("../data/entities.tsv", sep="\t")
grounded = pd.read_csv("../results/grounded_molecules_20_test.tsv", sep="\t")

In [998]:
entities

,entity,category,json_file
0,popc,MOL,zenodo_34415.json
1,amber,FFM,zenodo_34415.json
2,lipid14,FFM,zenodo_34415.json
3,cacl2,MOL,zenodo_34415.json
4,popc,MOL,zenodo_34415.json
...,...,...,...
1796,peggo,MOL,figshare_11894358.json
1797,peg,MOL,figshare_11894358.json
1798,dox,MOL,figshare_11894358.json
1799,dox,MOL,figshare_11894358.json


In [999]:
grounded

,MOL,MOL_TYPE,ERRORS,MOL_ID,MOL_SCORE,MOL_FULL_NAME,NB_results
0,popc,CHEBI,No errors,73001,38.63552,1-hexadecanoyl-2-(9Z-octadecenoyl)-sn-glycero-...,1
1,popc,CHEBI,No errors,73001,38.63552,1-hexadecanoyl-2-(9Z-octadecenoyl)-sn-glycero-...,1
2,cacl2,CHEBI,No errors,3312,39.09294,calcium dichloride,1
3,cacl2,CHEBI,No errors,3312,39.09294,calcium dichloride,1
4,dmpc,CHEBI,No errors,241349,39.09294,dimyristoyl phosphatidylcholine,2
...,...,...,...,...,...,...,...
733,osdas,Unknown,Unrecognized sequence,Not Available,Not Available,Not Available,Not Available
734,osda,Unknown,Unrecognized sequence,Not Available,Not Available,Not Available,Not Available
735,osda,Unknown,Unrecognized sequence,Not Available,Not Available,Not Available,Not Available
736,peggo,Unknown,Unrecognized sequence,Not Available,Not Available,Not Available,Not Available


In [1000]:
# We only select the entities that are molecules
mol_entities = entities[entities["category"] == "MOL"]

In [1001]:
mol_entities

,entity,category,json_file
0,popc,MOL,zenodo_34415.json
3,cacl2,MOL,zenodo_34415.json
4,popc,MOL,zenodo_34415.json
5,cacl2,MOL,zenodo_34415.json
11,popc,MOL,zenodo_34415.json
...,...,...,...
1796,peggo,MOL,figshare_11894358.json
1797,peg,MOL,figshare_11894358.json
1798,dox,MOL,figshare_11894358.json
1799,dox,MOL,figshare_11894358.json


In [1002]:
# We merge the two dataframes on the name of molecules
merged = mol_entities.merge(grounded, left_on="entity", right_on="MOL")

In [1003]:
merged

,entity,category,json_file,MOL,MOL_TYPE,ERRORS,MOL_ID,MOL_SCORE,MOL_FULL_NAME,NB_results
0,popc,MOL,zenodo_34415.json,popc,CHEBI,No errors,73001,38.63552,1-hexadecanoyl-2-(9Z-octadecenoyl)-sn-glycero-...,1
1,popc,MOL,zenodo_34415.json,popc,CHEBI,No errors,73001,38.63552,1-hexadecanoyl-2-(9Z-octadecenoyl)-sn-glycero-...,1
2,cacl2,MOL,zenodo_34415.json,cacl2,CHEBI,No errors,3312,39.09294,calcium dichloride,1
3,cacl2,MOL,zenodo_34415.json,cacl2,CHEBI,No errors,3312,39.09294,calcium dichloride,1
4,popc,MOL,zenodo_34415.json,popc,CHEBI,No errors,73001,38.63552,1-hexadecanoyl-2-(9Z-octadecenoyl)-sn-glycero-...,1
...,...,...,...,...,...,...,...,...,...,...
1935,peggo,MOL,figshare_11894358.json,peggo,Unknown,Unrecognized sequence,Not Available,Not Available,Not Available,Not Available
1936,peggo,MOL,figshare_11894358.json,peggo,Unknown,Unrecognized sequence,Not Available,Not Available,Not Available,Not Available
1937,peggo,MOL,figshare_11894358.json,peggo,Unknown,Unrecognized sequence,Not Available,Not Available,Not Available,Not Available
1938,peggo,MOL,figshare_11894358.json,peggo,Unknown,Unrecognized sequence,Not Available,Not Available,Not Available,Not Available


In [1004]:
# We only work with molecules were the grounding was succesfull
merged = merged[merged["MOL_ID"] != "Not Available"]

In [1005]:
merged

,entity,category,json_file,MOL,MOL_TYPE,ERRORS,MOL_ID,MOL_SCORE,MOL_FULL_NAME,NB_results
0,popc,MOL,zenodo_34415.json,popc,CHEBI,No errors,73001,38.63552,1-hexadecanoyl-2-(9Z-octadecenoyl)-sn-glycero-...,1
1,popc,MOL,zenodo_34415.json,popc,CHEBI,No errors,73001,38.63552,1-hexadecanoyl-2-(9Z-octadecenoyl)-sn-glycero-...,1
2,cacl2,MOL,zenodo_34415.json,cacl2,CHEBI,No errors,3312,39.09294,calcium dichloride,1
3,cacl2,MOL,zenodo_34415.json,cacl2,CHEBI,No errors,3312,39.09294,calcium dichloride,1
4,popc,MOL,zenodo_34415.json,popc,CHEBI,No errors,73001,38.63552,1-hexadecanoyl-2-(9Z-octadecenoyl)-sn-glycero-...,1
...,...,...,...,...,...,...,...,...,...,...
1923,doxorubicin,MOL,figshare_11894358.json,doxorubicin,CHEBI,No errors,64816,65.075066,doxorubicin(1+),11
1924,graphene oxide,MOL,figshare_11894358.json,graphene oxide,CHEBI,No errors,132889,52.538948,graphene oxide,1
1925,graphene oxide,MOL,figshare_11894358.json,graphene oxide,CHEBI,No errors,132889,52.538948,graphene oxide,1
1926,poly­(ethylene glycol),MOL,figshare_11894358.json,poly­(ethylene glycol),CHEBI,No errors,46793,24.279636,poly(ethylene glycol),1


In [1006]:
# We delete every duplicated row based on :
# - the mol id and type, the mol name and the json file
merged = merged.drop_duplicates(subset=["MOL_ID", "MOL_TYPE", "entity", "json_file"])


In [1007]:
merged

,entity,category,json_file,MOL,MOL_TYPE,ERRORS,MOL_ID,MOL_SCORE,MOL_FULL_NAME,NB_results
0,popc,MOL,zenodo_34415.json,popc,CHEBI,No errors,73001,38.63552,1-hexadecanoyl-2-(9Z-octadecenoyl)-sn-glycero-...,1
2,cacl2,MOL,zenodo_34415.json,cacl2,CHEBI,No errors,3312,39.09294,calcium dichloride,1
10,dmpc,MOL,figshare_8046437.json,dmpc,CHEBI,No errors,241349,39.09294,dimyristoyl phosphatidylcholine,2
18,dppc,MOL,zenodo_1009027.json,dppc,CHEBI,No errors,40265,40.728237,"1,2-di-O-palmitoyl-sn-glycero-3-phosphocholine",1
24,nacl,MOL,zenodo_1009027.json,nacl,CHEBI,No errors,26710,40.728237,sodium chloride,1
...,...,...,...,...,...,...,...,...,...,...
1898,alkylammonium,MOL,figshare_11916468.json,alkylammonium,CHEBI,No errors,38015,74.5974,alkylammonium sulfate,4
1912,tributylammonium,MOL,figshare_11916468.json,tributylammonium,PubChem,No errors,3724478,Not Available,tributylazanium,Not Available
1918,doxorubicin,MOL,figshare_11894358.json,doxorubicin,CHEBI,No errors,64816,65.075066,doxorubicin(1+),11
1920,graphene oxide,MOL,figshare_11894358.json,graphene oxide,CHEBI,No errors,132889,52.538948,graphene oxide,1


In [1008]:
# We want to count in how many json file we find the same grounding
# - We start by grouping the molecule based on their ID and TYPE
# - We apply the count method on the json_file column
#   to count in how many different json file we can find this grounding

counts = merged.groupby(["MOL_ID", "MOL_TYPE"])["json_file"].nunique()

In [1009]:
counts

MOL_ID    MOL_TYPE
10674918  PubChem     2
107786    PubChem     1
11822705  PubChem     1
12834     CHEBI       1
132889    CHEBI       2
                     ..
85365     CHEBI       1
91139     CHEBI       1
91189     CHEBI       1
91219     CHEBI       1
99479     PubChem     1
Name: json_file, Length: 189, dtype: int64

In [1010]:
# We reindex the columns so that MOL_ID and MOL_TYPE dont'appear as indexes
counts = counts.reset_index()

In [1011]:
# We rename the last column (number of json file wher we find the same grounding)
counts.columns = ["MOL_ID", "MOL_TYPE", "nb_json_files"]

In [1012]:
counts

,MOL_ID,MOL_TYPE,nb_json_files
0,10674918,PubChem,2
1,107786,PubChem,1
2,11822705,PubChem,1
3,12834,CHEBI,1
4,132889,CHEBI,2
...,...,...,...
184,85365,CHEBI,1
185,91139,CHEBI,1
186,91189,CHEBI,1
187,91219,CHEBI,1


In [1013]:
# We identify the grounding that have been found in more then one json_file
duplicates = counts[counts["nb_json_files"] > 1]

In [1014]:
duplicates

,MOL_ID,MOL_TYPE,nb_json_files
0,10674918,PubChem,2
4,132889,CHEBI,2
18,16113,CHEBI,10
20,16183,CHEBI,2
21,16199,CHEBI,2
35,17761,CHEBI,2
50,241349,CHEBI,4
58,26708,CHEBI,2
59,26710,CHEBI,16
73,3051,PubChem,2


In [1015]:
# We merge the duplicated data frame that contains the MOL_ID and MOL_TYPE that have
# been found in different json files with the merge dtafarame that contains all the
# information about the molecule grounded.
# We want to retreive the information about the molecule that have been found
# in different json file
details = merged.merge(duplicates[["MOL_ID", "MOL_TYPE"]], on=["MOL_ID", "MOL_TYPE"])

In [1016]:
details

,entity,category,json_file,MOL,MOL_TYPE,ERRORS,MOL_ID,MOL_SCORE,MOL_FULL_NAME,NB_results
0,popc,MOL,zenodo_34415.json,popc,CHEBI,No errors,73001,38.63552,1-hexadecanoyl-2-(9Z-octadecenoyl)-sn-glycero-...,1
1,dmpc,MOL,figshare_8046437.json,dmpc,CHEBI,No errors,241349,39.09294,dimyristoyl phosphatidylcholine,2
2,dppc,MOL,zenodo_1009027.json,dppc,CHEBI,No errors,40265,40.728237,"1,2-di-O-palmitoyl-sn-glycero-3-phosphocholine",1
3,nacl,MOL,zenodo_1009027.json,nacl,CHEBI,No errors,26710,40.728237,sodium chloride,1
4,popc,MOL,zenodo_1219494.json,popc,CHEBI,No errors,73001,38.63552,1-hexadecanoyl-2-(9Z-octadecenoyl)-sn-glycero-...,1
...,...,...,...,...,...,...,...,...,...,...
104,dopc,MOL,zenodo_573274.json,dopc,CHEBI,No errors,52360,40.728237,"1,2-dioleoyl-sn-glycero-3-phosphocholine(1+)",1
105,dlpc,MOL,zenodo_573274.json,dlpc,CHEBI,No errors,60273,37.084248,"1,2-dilauroyl-sn-glycero-3-phosphocholine(1+)",2
106,nacl,MOL,zenodo_573274.json,nacl,CHEBI,No errors,26710,40.728237,sodium chloride,1
107,nacl,MOL,figshare_4508975.json,nacl,CHEBI,No errors,26710,40.728237,sodium chloride,1


In [1017]:
# We only kepp intersting columns such as :
# - the mol_id the mol_type the json_file and the mol name (entity)
details = details[["MOL_ID", "MOL_TYPE", "json_file", "entity"]]

In [1018]:
details

,MOL_ID,MOL_TYPE,json_file,entity
0,73001,CHEBI,zenodo_34415.json,popc
1,241349,CHEBI,figshare_8046437.json,dmpc
2,40265,CHEBI,zenodo_1009027.json,dppc
3,26710,CHEBI,zenodo_1009027.json,nacl
4,73001,CHEBI,zenodo_1219494.json,popc
...,...,...,...,...
104,52360,CHEBI,zenodo_573274.json,dopc
105,60273,CHEBI,zenodo_573274.json,dlpc
106,26710,CHEBI,zenodo_573274.json,nacl
107,26710,CHEBI,figshare_4508975.json,nacl


In [1019]:
# We sort the data frame for a better readability
details = details.sort_values(["MOL_ID", "MOL_TYPE", "json_file"])

In [1020]:
details

,MOL_ID,MOL_TYPE,json_file,entity
9,10674918,PubChem,zenodo_51750.json,dmtap
39,10674918,PubChem,zenodo_53212.json,dmtap
108,132889,CHEBI,figshare_11894358.json,graphene oxide
76,132889,CHEBI,figshare_13836577.json,graphene oxide
55,16113,CHEBI,figshare_14994624.json,cholesterol
...,...,...,...,...
91,73001,CHEBI,zenodo_5362218.json,popc
79,73001,CHEBI,zenodo_6010416.json,popc
86,73001,CHEBI,zenodo_6817824.json,popc
57,73001,CHEBI,zenodo_6988344.json,popc


In [1021]:
# We save the dataframe in a tsv file
details.to_csv(
    "../results/same_grounding_molecules_filter_20_test.tsv", sep="\t", index=False
)

Part 2 : We create the knowledge graphs (grounded and ungroundes) for the molecules previously identied

In [1022]:
# We define a list of ions that we awant to exclude from our graph
IONS = {
    "na",
    "na+",
    "cl",
    "cl-",
    "nacl",
}

In [1023]:
# We create a function to format the json_file name :
# - exemple: zenodo_123456 to zenodo\n123456
def format_dataset(json_file: str) -> str:
    json_file = json_file.strip(".json")
    if "zenodo_" in json_file:
        json_file = json_file.replace("zenodo_", "zenodo\n")
    if "figshare_" in json_file:
        json_file = json_file.replace("figshare_", "figshare\n")
    return json_file


In [1024]:
# We create a list containning the json_file (with the previous function)
json_nodes = []
for _, row in details.iterrows():
    if row["entity"] not in IONS:
        json_nodes.append(format_dataset(row["json_file"]))

In [1025]:
json_nodes

['zenodo\n51750',
 'zenodo\n53212',
 'figshare\n11894358',
 'figshare\n13836577',
 'figshare\n14994624',
 'figshare\n4806544',
 'zenodo\n247386',
 'zenodo\n259443',
 'zenodo\n2645909',
 'zenodo\n2653735',
 'zenodo\n3988469',
 'zenodo\n4445375',
 'zenodo\n6010416',
 'zenodo\n6144286',
 'figshare\n12661589',
 'zenodo\n4106413',
 'figshare\n12661589',
 'zenodo\n6791876',
 'zenodo\n1219494',
 'zenodo\n3228177',
 'figshare\n14994624',
 'figshare\n8046437',
 'zenodo\n51750',
 'zenodo\n53212',
 'figshare\n14511885',
 'zenodo\n1198454',
 'zenodo\n3592499',
 'zenodo\n573274',
 'figshare\n11808396',
 'zenodo\n3975394',
 'figshare\n14994624',
 'figshare\n14994624',
 'figshare\n3426170',
 'zenodo\n3613573',
 'zenodo\n6817824',
 'figshare\n13836577',
 'zenodo\n3696970',
 'zenodo\n1009027',
 'zenodo\n1009607',
 'zenodo\n14591',
 'zenodo\n15550',
 'zenodo\n259443',
 'zenodo\n2645909',
 'zenodo\n3950029',
 'zenodo\n4445375',
 'zenodo\n6010416',
 'zenodo\n6817824',
 'zenodo\n838635',
 'figshare\n149946

In [1026]:
# We create two list one for the molecule name and one for the molecule IDS
# - We check every row to make sure that the molecule is not and ions
mol_nodes = []
mol_id_nodes = []
for _, row in details.iterrows():
    if row["entity"] not in IONS:
        mol_nodes.append(row["entity"])
        mol_id_nodes.append(row["MOL_TYPE"] + "\n" + row["MOL_ID"])

In [1027]:
mol_nodes

['dmtap',
 'dmtap',
 'graphene oxide',
 'graphene oxide',
 'cholesterol',
 'cholesterol',
 'cholesterol',
 'cholesterol',
 'cholesterol',
 'cholesterol',
 'cholesterol',
 'cholesterol',
 'cholesterol',
 'cholesterol',
 'methane',
 'methane',
 'urea',
 'urea',
 'ceramide',
 'ceramide',
 'dmpc',
 'dmpc',
 'dmpc',
 'dmpc',
 'sodium',
 'sodium',
 'depc',
 'depc',
 'lopinavir',
 'lopinavir',
 'tio2',
 'titanium dioxide',
 'tio2',
 'popg',
 'popg',
 'graphene',
 'graphene',
 'dppc',
 'dppc',
 'dppc',
 'dppc',
 'dppc',
 'dppc',
 'dppc',
 'dppc',
 'dppc',
 'dppc',
 'dppc',
 '1,2-dimyristoyl-sn-glycero-3-phosphocholine',
 'dimyristoylphosphatidylcholine',
 'dimyristoylphosphatidylcholine',
 'dopc',
 'dopc',
 'dopc',
 'dopc',
 'dopc',
 'dlpc',
 'dlpc',
 'pope',
 'pope',
 'pope',
 'pops',
 'phosphatidylcholine',
 'phosphatidylcholine',
 'amyloid-β',
 'amyloid beta',
 'amyloid beta',
 'popc',
 'popc',
 'popc',
 'popc',
 'popc',
 'popc',
 'popc',
 'popc',
 'popc',
 'popc',
 'popc',
 'popc',
 'popc'

In [1028]:
# We create edges between the molecule names and the json file it has been found in.
dataset_mol_edges = []
for _, row in details.iterrows():
    if row["entity"] not in IONS:
        dataset_mol_edges.append({format_dataset(row["json_file"]), row["entity"]})

In [1029]:
# We create edges between the molecule names and the molecule id.
mol_molids_edges = []
for _, row in details.iterrows():
    if row["entity"] not in IONS:
        mol_molids_edges.append({row["MOL_TYPE"] + "\n" + row["MOL_ID"], row["entity"]})

In [1030]:
# We create an ungrounded knowledge graph that has:
# - Nodes: the json_file name and the molecule name
# - edges: link between the json file and its molecules
knowledge_graph = nx.Graph()

knowledge_graph.add_nodes_from(json_nodes, color="#f8ed62")
knowledge_graph.add_nodes_from(mol_nodes, color="skyblue")

knowledge_graph.add_edges_from(dataset_mol_edges)

In [1031]:
mol_nodes


['dmtap',
 'dmtap',
 'graphene oxide',
 'graphene oxide',
 'cholesterol',
 'cholesterol',
 'cholesterol',
 'cholesterol',
 'cholesterol',
 'cholesterol',
 'cholesterol',
 'cholesterol',
 'cholesterol',
 'cholesterol',
 'methane',
 'methane',
 'urea',
 'urea',
 'ceramide',
 'ceramide',
 'dmpc',
 'dmpc',
 'dmpc',
 'dmpc',
 'sodium',
 'sodium',
 'depc',
 'depc',
 'lopinavir',
 'lopinavir',
 'tio2',
 'titanium dioxide',
 'tio2',
 'popg',
 'popg',
 'graphene',
 'graphene',
 'dppc',
 'dppc',
 'dppc',
 'dppc',
 'dppc',
 'dppc',
 'dppc',
 'dppc',
 'dppc',
 'dppc',
 'dppc',
 '1,2-dimyristoyl-sn-glycero-3-phosphocholine',
 'dimyristoylphosphatidylcholine',
 'dimyristoylphosphatidylcholine',
 'dopc',
 'dopc',
 'dopc',
 'dopc',
 'dopc',
 'dlpc',
 'dlpc',
 'pope',
 'pope',
 'pope',
 'pops',
 'phosphatidylcholine',
 'phosphatidylcholine',
 'amyloid-β',
 'amyloid beta',
 'amyloid beta',
 'popc',
 'popc',
 'popc',
 'popc',
 'popc',
 'popc',
 'popc',
 'popc',
 'popc',
 'popc',
 'popc',
 'popc',
 'popc'

In [1032]:
# We create a grounded knowledge graph that has:
# - Nodes: - the json_file name
#          - the molecule names
#          - the molecule ids
# - edges: - link between the json file and its molecules
#          - link between the molecule and its grounding id
knowledge_graph_grounded = nx.Graph()

knowledge_graph_grounded.add_nodes_from(json_nodes, color="#f8ed62")
knowledge_graph_grounded.add_nodes_from(mol_nodes, color="skyblue")
knowledge_graph_grounded.add_nodes_from(mol_id_nodes, color="#ffbaba")

knowledge_graph_grounded.add_edges_from(dataset_mol_edges)
knowledge_graph_grounded.add_edges_from(mol_molids_edges)
grounding_edges = list(mol_molids_edges)


In [1033]:
dataset_mol_edges

[{'dmtap', 'zenodo\n51750'},
 {'dmtap', 'zenodo\n53212'},
 {'figshare\n11894358', 'graphene oxide'},
 {'figshare\n13836577', 'graphene oxide'},
 {'cholesterol', 'figshare\n14994624'},
 {'cholesterol', 'figshare\n4806544'},
 {'cholesterol', 'zenodo\n247386'},
 {'cholesterol', 'zenodo\n259443'},
 {'cholesterol', 'zenodo\n2645909'},
 {'cholesterol', 'zenodo\n2653735'},
 {'cholesterol', 'zenodo\n3988469'},
 {'cholesterol', 'zenodo\n4445375'},
 {'cholesterol', 'zenodo\n6010416'},
 {'cholesterol', 'zenodo\n6144286'},
 {'figshare\n12661589', 'methane'},
 {'methane', 'zenodo\n4106413'},
 {'figshare\n12661589', 'urea'},
 {'urea', 'zenodo\n6791876'},
 {'ceramide', 'zenodo\n1219494'},
 {'ceramide', 'zenodo\n3228177'},
 {'dmpc', 'figshare\n14994624'},
 {'dmpc', 'figshare\n8046437'},
 {'dmpc', 'zenodo\n51750'},
 {'dmpc', 'zenodo\n53212'},
 {'figshare\n14511885', 'sodium'},
 {'sodium', 'zenodo\n1198454'},
 {'depc', 'zenodo\n3592499'},
 {'depc', 'zenodo\n573274'},
 {'figshare\n11808396', 'lopinavir'}

In [1034]:
mol_molids_edges

[{'PubChem\n10674918', 'dmtap'},
 {'PubChem\n10674918', 'dmtap'},
 {'CHEBI\n132889', 'graphene oxide'},
 {'CHEBI\n132889', 'graphene oxide'},
 {'CHEBI\n16113', 'cholesterol'},
 {'CHEBI\n16113', 'cholesterol'},
 {'CHEBI\n16113', 'cholesterol'},
 {'CHEBI\n16113', 'cholesterol'},
 {'CHEBI\n16113', 'cholesterol'},
 {'CHEBI\n16113', 'cholesterol'},
 {'CHEBI\n16113', 'cholesterol'},
 {'CHEBI\n16113', 'cholesterol'},
 {'CHEBI\n16113', 'cholesterol'},
 {'CHEBI\n16113', 'cholesterol'},
 {'CHEBI\n16183', 'methane'},
 {'CHEBI\n16183', 'methane'},
 {'CHEBI\n16199', 'urea'},
 {'CHEBI\n16199', 'urea'},
 {'CHEBI\n17761', 'ceramide'},
 {'CHEBI\n17761', 'ceramide'},
 {'CHEBI\n241349', 'dmpc'},
 {'CHEBI\n241349', 'dmpc'},
 {'CHEBI\n241349', 'dmpc'},
 {'CHEBI\n241349', 'dmpc'},
 {'CHEBI\n26708', 'sodium'},
 {'CHEBI\n26708', 'sodium'},
 {'PubChem\n3051', 'depc'},
 {'PubChem\n3051', 'depc'},
 {'CHEBI\n31781', 'lopinavir'},
 {'CHEBI\n31781', 'lopinavir'},
 {'CHEBI\n32234', 'tio2'},
 {'CHEBI\n32234', 'titani

In [1035]:
# We create a function that will relabel long molecule names
# - exemple: 1,2-dimyristoyl-sn-glycero-3-phosphocholine -> 1,2-di...
def get_molecule_label(nodes):
    d = {}
    for n in nodes:
        d[n] = n if len(n) <= 10 else n[:10] + "..."
    return d


In [1036]:
# We use a visualisation function that will display the knowledge graphes
def visualize_graph(G, mol_nodes, out, red_edges=None):

    node_colors = [G.nodes[n].get("color", "skyblue") for n in G.nodes]

    label_map = get_molecule_label(mol_nodes)
    labels = {n: label_map.get(n, n) for n in G.nodes}

    pos = nx.spring_layout(G, k=0.2)

    plt.figure(figsize=(22, 18))

    nx.draw(
        G,
        pos,
        labels=labels,
        node_color=node_colors,
        node_size=1000,
        font_size=10,
        edge_color="gray",
        linewidths=4,
        alpha=1,
    )
    # This part will allow us to highligth in red the new link made by the grounding
    if red_edges:
        nx.draw_networkx_edges(
            G,
            pos,
            edgelist=(red_edges),
            edge_color="red",
            width=3,
        )

    plt.title("Knowledge Graph")
    plt.savefig(out)
    plt.close()

    logger.info("Saved:", out)

In [1037]:
# We create a list containing the list of the new edges created with the grounding
new_edges = (knowledge_graph_grounded.edges()) - (knowledge_graph.edges())

In [ ]:
visualize_graph(
    knowledge_graph, mol_nodes, "filter_non_grounded_20_test.png", red_edges=None
)

visualize_graph(
    knowledge_graph_grounded,
    mol_nodes,
    "grounded_graph_filter_20_test.png",
    red_edges=new_edges,
)

2026-05-07 15:08:47.751 | INFO     | __main__:visualize_graph:38 - Saved:
2026-05-07 15:08:48.226 | INFO     | __main__:visualize_graph:38 - Saved:


In [1039]:
# We create a function that will print the statistics of the graphes
def print_graph_stats(knowledge_graph: nx.Graph) -> None:
    """Print basic statistics about the graph."""
    num_components = nx.number_connected_components(knowledge_graph)
    logger.info(f"Number of nodes: {knowledge_graph.number_of_nodes()}")
    logger.info(f"Density: {nx.density(knowledge_graph)}")
    logger.info(f"Number of edges: {knowledge_graph.number_of_edges()}")
    logger.info(f"Number of disjoint subgraphs: {num_components}")
    logger.info("CONNECTED COMPONENT:")
    for i, component in enumerate(list(nx.connected_components(knowledge_graph))):
        logger.info(f"Connected components {i + 1}:{component}")


In [1040]:
print_graph_stats(knowledge_graph)

2026-05-07 15:08:48.242 | INFO     | __main__:print_graph_stats:5 - Number of nodes: 80
2026-05-07 15:08:48.243 | INFO     | __main__:print_graph_stats:6 - Density: 0.029430379746835444
2026-05-07 15:08:48.244 | INFO     | __main__:print_graph_stats:7 - Number of edges: 93
2026-05-07 15:08:48.245 | INFO     | __main__:print_graph_stats:8 - Number of disjoint subgraphs: 5
2026-05-07 15:08:48.245 | INFO     | __main__:print_graph_stats:9 - CONNECTED COMPONENT:
2026-05-07 15:08:48.246 | INFO     | __main__:print_graph_stats:11 - Connected components 1:{'dmtap', 'zenodo\n13814', 'dimyristoylphosphatidylcholine', 'zenodo\n1009027', 'zenodo\n1198454', 'dlpc', 'zenodo\n13853', 'zenodo\n1009607', 'zenodo\n34415', 'depc', 'popc', 'zenodo\n6988344', 'figshare\n14511885', 'zenodo\n51750', 'tio2', 'zenodo\n247386', 'figshare\n4806544', 'cholesterol', 'zenodo\n1293762', 'zenodo\n51185', 'sodium', 'zenodo\n2645909', 'popg', 'zenodo\n3228177', 'zenodo\n259443', 'zenodo\n3592499', 'zenodo\n5226209', '

In [1041]:
print_graph_stats(knowledge_graph_grounded)

2026-05-07 15:08:48.257 | INFO     | __main__:print_graph_stats:5 - Number of nodes: 101
2026-05-07 15:08:48.259 | INFO     | __main__:print_graph_stats:6 - Density: 0.023366336633663366
2026-05-07 15:08:48.260 | INFO     | __main__:print_graph_stats:7 - Number of edges: 118
2026-05-07 15:08:48.260 | INFO     | __main__:print_graph_stats:8 - Number of disjoint subgraphs: 5
2026-05-07 15:08:48.261 | INFO     | __main__:print_graph_stats:9 - CONNECTED COMPONENT:
2026-05-07 15:08:48.261 | INFO     | __main__:print_graph_stats:11 - Connected components 1:{'zenodo\n1009027', 'zenodo\n1009607', 'zenodo\n34415', 'popc', 'zenodo\n6988344', 'CHEBI\n52360', 'CHEBI\n241349', 'zenodo\n51185', 'CHEBI\n26708', 'popg', 'zenodo\n30904', 'zenodo\n14591', 'pops', 'zenodo\n1293813', 'zenodo\n1118682', 'zenodo\n5362218', 'CHEBI\n73001', 'CHEBI\n60286', 'figshare\n8046437', 'dopc', 'zenodo\n53212', 'zenodo\n6010416', 'dppc', 'zenodo\n1219494', 'zenodo\n3950029', 'zenodo\n13814', 'CHEBI\n64482', 'zenodo\n11